# Ensemble of ConvNeXt and EfficientNetV2-S

This notebook combines two completed runs without retraining: Exp1 (standard ConvNeXt-base, public score 0.97003) and Exp3 (EfficientNetV2-S, public score 0.97015). They score almost the same on the leaderboard but belong to different CNN families, and on the test set they disagree in 219 label cells, 3.3 times as many as Exp1 and Exp2 (66). Averaging models that make different mistakes is where an ensemble can help, unlike the Exp1 + Exp2 mean (0.96824). 

By default the notebook only reads saved probabilities and scores them. Test inference runs only if a member has no saved test probabilities and `STAGE2_RUN_TEST_INFERENCE=1`. The submission CSV and the run record are written only with `BUILD_SUBMISSION=True`. Kaggle upload is always manual.

In [1]:
from __future__ import annotations

import json
import os
import platform
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import torch

RUN_TEST_INFERENCE = os.environ.get('STAGE2_RUN_TEST_INFERENCE', '0') == '1'
BUILD_SUBMISSION = True  # Also saves the ensemble's fold-0 probabilities and run record.
SUBMIT_TO_KAGGLE = False  # No submission API call exists in this notebook.

RUN_ID = 'ensemble_exp1_exp3_mean_fold0'
VALIDATION_FOLD = 0
THRESHOLD = 0.25
TEST_BATCH_SIZE = 8
# Experiment folder -> completed run ID. Add a member here once its fold-0 run exists.
MEMBERS = {
    'exp1': 'convnext_standard_fold0_seed42_20261018_full1',
    'exp3': 'efficientnetv2s_fold0_seed42',
}

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print('Device:', DEVICE, '| test inference:', RUN_TEST_INFERENCE, '| build submission:', BUILD_SUBMISSION)

Device: mps | test inference: False | build submission: True


## Load the members' fold-0 predictions

Each completed run saved its probabilities for the 2,452 fold-0 images. We join them by image ID, re-score them, and require the result to match the score in the run record. The members must share the fold file, label order and threshold. Otherwise the comparison would not be on equal terms. Exp3 was trained on the same folds 1–4, but from its own head warm-up rather than Exp0's head.

In [2]:
start = Path(os.environ.get('STAGE2_ROOT', Path.cwd())).resolve()
ROOT = next((path for path in (start, *start.parents) if (path / 'folds.csv').is_file() and (path / 'stage2' / 'experiment.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Launch Jupyter from this repository or a subdirectory.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from stage2.ensemble import load_finetuned_network, mean_probabilities, read_probabilities, write_test_probabilities
from stage2.experiment import LABEL_COLUMNS, ImageDataset, form_submission, read_csv, score_probabilities, split_rows, write_probabilities, write_run_record, write_submission
from stage2.run_records import file_sha256
from stage2.train import predict_probabilities

DATA_DIR = ROOT / 'data'
_, validation_rows = split_rows(DATA_DIR, ROOT / 'folds.csv', VALIDATION_FOLD)
validation_ids = [row['image_id'] for row in validation_rows]
validation_targets = np.asarray([[int(row[label]) for label in LABEL_COLUMNS] for row in validation_rows])
fold_sha256 = file_sha256(ROOT / 'folds.csv')

records, checkpoints, validation_probabilities = {}, {}, {}
for name, run_id in MEMBERS.items():
    run_dir = ROOT / 'experiments' / name / 'results' / run_id
    record = json.loads((run_dir / f'{run_id}.json').read_text())
    if record['status'] != 'completed' or record['fold'] != VALIDATION_FOLD:
        raise ValueError(f'{run_id} is not a completed fold-{VALIDATION_FOLD} run.')
    if record['data']['fold_sha256'] != fold_sha256 or record['data']['label_columns'] != list(LABEL_COLUMNS):
        raise ValueError(f'{run_id} used a different fold file or label order.')
    if record['metrics']['threshold'] != THRESHOLD:
        raise ValueError(f'{run_id} was scored at another threshold.')
    probabilities = read_probabilities(run_dir / f'{run_id}_val.csv', validation_ids)
    rescored = score_probabilities(validation_targets, probabilities, THRESHOLD)['macro_f1']
    if not np.isclose(rescored, record['metrics']['macro_f1'], rtol=0, atol=1e-9):
        raise ValueError(f'{run_id}: re-scored macro F1 {rescored} differs from its record.')
    records[name] = record
    checkpoints[name] = ROOT / 'experiments' / name / 'checkpoints' / f'{run_id}_best.ckpt'
    validation_probabilities[name] = probabilities
    print(f'{name}: {run_id} | fold-0 macro F1 {rescored:.5f} (matches record)')

exp1: convnext_standard_fold0_seed42_20261018_full1 | fold-0 macro F1 0.97518 (matches record)
exp3: efficientnetv2s_fold0_seed42 | fold-0 macro F1 0.99831 (matches record)


## Compare single models and their mean on fold 0

We average probabilities with equal weights and keep the 0.25 threshold. We do not tune weights or thresholds: fold 0 has only 8–12 photos of each named pet, so a tuned weight would fit noise. On fold 0 one named-pet photo moves macro F1 by about 0.012 (each pet has only 8–10 validation photos), so differences smaller than that are not real differences. Exp3's fold-0 score (0.99831) was selected on fold 0 and is optimistic: its leaderboard score is level with Exp1's, so we expect the fold-0 table to favour Exp3 more than the test set does.

The two members use different architectures, so their errors should overlap less than Exp1's and Exp2's did. The disagreement count shows how much they differ. An ensemble cannot fix an image that every member gets wrong.

In [3]:
candidates = {**validation_probabilities, 'mean': mean_probabilities(list(validation_probabilities.values()))}
scores = {name: score_probabilities(validation_targets, probabilities, THRESHOLD) for name, probabilities in candidates.items()}
scores_at_half = {name: score_probabilities(validation_targets, probabilities, 0.5) for name, probabilities in candidates.items()}

header = f"{'model':<6} {'F1@0.25':>8} " + ' '.join(f'{label:>8}' for label in LABEL_COLUMNS) + f" {'AP':>8} {'F1@0.5':>8}"
print(header)
print('-' * len(header))
for name, metrics in scores.items():
    per_label = ' '.join(f"{metrics['per_label_f1'][label]:>8.4f}" for label in LABEL_COLUMNS)
    print(f"{name:<6} {metrics['macro_f1']:>8.5f} {per_label} {metrics['macro_ap']:>8.5f} {scores_at_half[name]['macro_f1']:>8.5f}")

predictions = {name: probabilities >= THRESHOLD for name, probabilities in candidates.items()}
member_names = list(MEMBERS)
disagreement = np.any([predictions[name] != predictions[member_names[0]] for name in member_names[1:]], axis=0)
print('\nImage-label cells where the members disagree:', int(disagreement.sum()),
      {label: int(disagreement[:, index].sum()) for index, label in enumerate(LABEL_COLUMNS)})
for name in candidates:
    false_positives = (predictions[name] & ~validation_targets.astype(bool)).sum(axis=0)
    false_negatives = (~predictions[name] & validation_targets.astype(bool)).sum(axis=0)
    print(f'{name:<6} errors per label (FP/FN):', {label: f'{fp}/{fn}' for label, fp, fn in zip(LABEL_COLUMNS, false_positives, false_negatives)})

model   F1@0.25      Cat      Dog    Moris    Motya  Biatrix       AP   F1@0.5
------------------------------------------------------------------------------
exp1    0.97518   0.9966   0.9970   1.0000   0.9412   0.9412  0.99443  0.97509
exp3    0.99831   0.9971   0.9945   1.0000   1.0000   1.0000  0.99998  0.99832
mean    0.98646   0.9961   0.9950   1.0000   1.0000   0.9412  0.99999  0.99871

Image-label cells where the members disagree: 22 {'Cat': 5, 'Dog': 15, 'Moris': 0, 'Motya': 1, 'Biatrix': 1}
exp1   errors per label (FP/FN): {'Cat': '5/2', 'Dog': '4/2', 'Moris': '0/0', 'Motya': '0/1', 'Biatrix': '1/0'}
exp3   errors per label (FP/FN): {'Cat': '5/1', 'Dog': '6/5', 'Moris': '0/0', 'Motya': '0/0', 'Biatrix': '0/0'}
mean   errors per label (FP/FN): {'Cat': '7/1', 'Dog': '9/1', 'Moris': '0/0', 'Motya': '0/0', 'Biatrix': '1/0'}


### Named-pet mistakes

Every fold-0 image where at least one model makes a named-pet mistake, with each model's probability. This shows whether the mean rescues a member's miss or adds a new one.

In [4]:
named_pets = [LABEL_COLUMNS.index(label) for label in ('Moris', 'Motya', 'Biatrix')]
print(f"{'image_id':<14} {'label':<8} {'true':>4} " + ' '.join(f'{name:>7}' for name in candidates))
for row_index, image_id in enumerate(validation_ids):
    for label_index in named_pets:
        truth = bool(validation_targets[row_index, label_index])
        if any(predictions[name][row_index, label_index] != truth for name in candidates):
            values = ' '.join(f'{candidates[name][row_index, label_index]:>7.3f}' for name in candidates)
            print(f'{image_id[:12]:<14} {LABEL_COLUMNS[label_index]:<8} {int(truth):>4} {values}')

image_id       label    true    exp1    exp3    mean
5be84f949fb2   Motya       1   0.024   1.000   0.512
520c9a44013a   Biatrix     0   0.642   0.174   0.408


## Reuse the saved test probabilities

Each member's test probabilities are saved next to its run as `experiments/<exp>/results/<run-id>_test/` (the earlier submissions kept only 0/1 labels, which cannot be averaged). Both Exp1 and Exp3 already have them, so this cell reads them and runs no inference. A cached file is reused only if its record carries the same checkpoint hash. If a file were missing, the member would be predicted again, which needs `STAGE2_RUN_TEST_INFERENCE=1` and the member's checkpoint.

As a reproducibility check, we threshold each member's probabilities at 0.25 and compare them with the CSV that was scored on Kaggle. Exp1 matched in all 93,365 cells when we checked it earlier. A few differing cells are expected only from arithmetic differences near the threshold.

In [5]:
test_rows = read_csv(DATA_DIR / 'test.csv', ('image_id',))
test_ids = [row['image_id'] for row in test_rows]
git_commit = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=ROOT, capture_output=True, text=True, check=True).stdout.strip() if (ROOT / '.git').exists() else None
source_files = (ROOT / 'notebooks' / '10_ensemble_exp1_exp3.ipynb', ROOT / 'stage2' / 'ensemble.py', ROOT / 'stage2' / 'experiment.py')
source_sha256 = {str(path.relative_to(ROOT)): file_sha256(path) for path in source_files}

test_probabilities = {}
for name, run_id in MEMBERS.items():
    record = records[name]
    cache_dir = ROOT / 'experiments' / name / 'results' / f'{run_id}_test'
    cache_csv, cache_json = cache_dir / f'{run_id}_test.csv', cache_dir / f'{run_id}_test.json'
    if cache_csv.is_file():
        cached = json.loads(cache_json.read_text())
        if cached['source_checkpoint_sha256'] != record['artifacts']['best_checkpoint_sha256']:
            raise ValueError(f'{cache_csv} was made from another checkpoint.')
        test_probabilities[name] = read_probabilities(cache_csv, test_ids)
        print(f'{name}: reused saved test probabilities.')
        continue
    if not RUN_TEST_INFERENCE:
        print(f'{name}: no saved test probabilities; set STAGE2_RUN_TEST_INFERENCE=1 to predict them.')
        continue
    from torch.utils.data import DataLoader

    network = load_finetuned_network(record, checkpoints[name])
    test_data = ImageDataset(test_rows, DATA_DIR / 'test', record['data']['image_size'], record['data']['mean'], record['data']['std'])
    test_loader = DataLoader(test_data, batch_size=TEST_BATCH_SIZE, shuffle=False, num_workers=0)
    print(f'{name}: predicting {len(test_rows)} test images on {DEVICE}...', flush=True)
    started = time.perf_counter()
    with torch.inference_mode():
        predicted_ids, probabilities = predict_probabilities(network, test_loader, DEVICE)
    seconds = time.perf_counter() - started
    if predicted_ids != test_ids:
        raise ValueError('Test loader changed image order.')
    write_test_probabilities(cache_csv, predicted_ids, probabilities)
    write_run_record(cache_json, {
        'run_id': f'{run_id}_test', 'experiment': record['experiment'], 'status': 'test_predicted',
        'source_run_id': run_id, 'source_checkpoint_sha256': record['artifacts']['best_checkpoint_sha256'],
        'test_images': len(predicted_ids), 'transform': record['data']['transform'] + '; no test-time augmentation',
        'git_commit': git_commit, 'source_sha256': source_sha256, 'device': str(DEVICE), 'prediction_seconds': seconds,
        'versions': {'python': platform.python_version(), 'torch': torch.__version__},
    })
    test_probabilities[name] = probabilities
    print(f'{name}: saved {cache_csv.relative_to(ROOT)} in {seconds / 60:.1f} min.')
    del network
    if DEVICE.type == 'mps':
        torch.mps.empty_cache()
    elif DEVICE.type == 'cuda':
        torch.cuda.empty_cache()

scored_submissions = {'exp1': 'exp1_convnext_standard_fold0_025.csv', 'exp3': 'exp3_efficientnetv2s_fold0_seed42_025.csv'}
for name, probabilities in test_probabilities.items():
    path = ROOT / 'submissions' / scored_submissions.get(name, '')
    if path.is_file():
        scored = read_probabilities(path, test_ids)
        changed = (probabilities >= THRESHOLD) != scored.astype(bool)
        print(f'{name}: {int(changed.sum())} of {changed.size} cells differ from {path.name}',
              {label: int(changed[:, index].sum()) for index, label in enumerate(LABEL_COLUMNS)})

exp1: reused saved test probabilities.
exp3: reused saved test probabilities.
exp1: 0 of 93365 cells differ from exp1_convnext_standard_fold0_025.csv {'Cat': 0, 'Dog': 0, 'Moris': 0, 'Motya': 0, 'Biatrix': 0}
exp3: 0 of 93365 cells differ from exp3_efficientnetv2s_fold0_seed42_025.csv {'Cat': 0, 'Dog': 0, 'Moris': 0, 'Motya': 0, 'Biatrix': 0}


## Form the ensemble submission

The ensemble's test probabilities are the equal-weight mean of the members', thresholded at 0.25. The table shows how many positives each label gets. A large jump in named-pet counts would need a closer look before submission. The cell also counts how many test cells the mean changes compared with each member. If it changes only a handful, a submission cannot move the score; if it changes dozens, the leaderboard decides whether the changes are right.

With `BUILD_SUBMISSION=True`, the notebook writes the CSV to `submissions/`, plus the ensemble's fold-0 probabilities and a run record under `experiments/exp4/results/<run-id>/`. Nothing is overwritten. Record the public score in the README after a manual upload.

In [6]:
if len(test_probabilities) == len(MEMBERS):
    ensemble_test = mean_probabilities([test_probabilities[name] for name in MEMBERS])
    print(f"{'labels':<8} " + ' '.join(f'{name:>7}' for name in (*MEMBERS, 'mean')))
    for index, label in enumerate(LABEL_COLUMNS):
        counts = [int((test_probabilities[name][:, index] >= THRESHOLD).sum()) for name in MEMBERS]
        counts.append(int((ensemble_test[:, index] >= THRESHOLD).sum()))
        print(f'{label:<8} ' + ' '.join(f'{count:>7}' for count in counts))
    ensemble_labels = ensemble_test >= THRESHOLD
    for name in MEMBERS:
        changed = ensemble_labels != (test_probabilities[name] >= THRESHOLD)
        print(f'Mean differs from {name} in {int(changed.sum())} cells:', {label: int(changed[:, index].sum()) for index, label in enumerate(LABEL_COLUMNS)})
else:
    ensemble_test = None
    print('Test probabilities are missing for some members; no ensemble test predictions yet.')

if BUILD_SUBMISSION:
    if ensemble_test is None:
        raise ValueError('Predict every member on the test set before building a submission.')
    result_dir = ROOT / 'experiments' / 'exp4' / 'results' / RUN_ID
    submission_path = ROOT / 'submissions' / f"exp4_{RUN_ID.removesuffix('_fold0')}_025.csv"
    for path in (submission_path, result_dir / f'{RUN_ID}.json', result_dir / f'{RUN_ID}_val.csv'):
        if path.exists():
            raise FileExistsError(f'Refusing to overwrite {path}')
    write_submission(submission_path, form_submission(DATA_DIR / 'sample_submission.csv', test_ids, ensemble_test, THRESHOLD))
    write_probabilities(result_dir / f'{RUN_ID}_val.csv', validation_ids, candidates['mean'], VALIDATION_FOLD)
    write_run_record(result_dir / f'{RUN_ID}.json', {
        'run_id': RUN_ID, 'experiment': '04_ensemble', 'status': 'completed', 'fold': VALIDATION_FOLD,
        'method': 'equal-weight mean of member probabilities, then fixed threshold',
        'members': [{'experiment': records[name]['experiment'], 'run_id': run_id,
                     'checkpoint_sha256': records[name]['artifacts']['best_checkpoint_sha256'],
                     'fold0_macro_f1': scores[name]['macro_f1']} for name, run_id in MEMBERS.items()],
        'data': {'fold_file': 'folds.csv', 'fold_sha256': fold_sha256, 'label_columns': list(LABEL_COLUMNS)},
        'metrics': {**scores['mean'], 'macro_f1_at_0_5': scores_at_half['mean']['macro_f1']},
        'artifacts': {'validation_probabilities_csv': str((result_dir / f'{RUN_ID}_val.csv').relative_to(ROOT)),
                      'submission_csv': str(submission_path.relative_to(ROOT))},
        'git_commit': git_commit, 'source_sha256': source_sha256,
        'public_leaderboard_score': None, 'error': None,
    })
    print('CSV ready for manual review:', submission_path.relative_to(ROOT))
    print('Run record:', (result_dir / f'{RUN_ID}.json').relative_to(ROOT))
else:
    print('No submission CSV built or uploaded.')

labels      exp1    exp3    mean
Cat         7781    7809    7815
Dog         7675    7659    7699
Moris        162     161     171
Motya        129     129     131
Biatrix      187     181     187
Mean differs from exp1 in 123 cells: {'Cat': 46, 'Dog': 36, 'Moris': 17, 'Motya': 14, 'Biatrix': 10}
Mean differs from exp3 in 96 cells: {'Cat': 22, 'Dog': 50, 'Moris': 12, 'Motya': 4, 'Biatrix': 8}
CSV ready for manual review: submissions/exp4_ensemble_exp1_exp3_mean_025.csv
Run record: experiments/exp4/results/ensemble_exp1_exp3_mean_fold0/ensemble_exp1_exp3_mean_fold0.json


## Results and conclusions

| Model | Fold-0 macro F1 at 0.25 | Public leaderboard |
|---|---:|---:|
| Exp1: ConvNeXt-base | 0.97518 | 0.97003 |
| Exp3: EfficientNetV2-S | 0.99831 | 0.97015 |
| **Mean of Exp1 and Exp3** (this notebook) | 0.98646 | **0.97761** |
| *For contrast: mean of Exp1 and Exp2* | 0.97528 | 0.96824 |

Public scores come from the Kaggle submission list, entered by hand. Fold 0 is a screening result, not five-fold CV.

**1. The ensemble clearly beats both of its members.** 0.97761 is +0.0075 over Exp3 and +0.0076 over Exp1. At the test set's counts one named-pet photo is worth roughly 0.0007 macro F1, so this is on the order of ten photos, well above the 0.001–0.002 band we set as noise before submitting. This is the best public score we have for now. It is still one submission scored on the public part of the test set, and we chose it because of that score, so the private score may be somewhat lower.

**2. Why it worked here and not for Exp1 + Exp2.** Exp1 and Exp3 are different CNN families with almost the same leaderboard score, and they disagree in 219 test label cells (Exp1 and Exp2: 66). The mean changed 123 cells compared with Exp1 and 96 compared with Exp3, and the leaderboard shows those changes were mostly correct. Exp3 recovers a fold-0 Motya photo that both ConvNeXt models missed (probability 1.000 against 0.024 and 0.078). The Exp1 + Exp2 mean, built from two models that share a backbone and recipe, lost 0.0018 against Exp1. So the diversity of the members mattered more than their number.

**3. Fold 0 could not predict this.** On fold 0 the mean (0.98646) ranked below Exp3 (0.99831): it inherited one Biatrix false alarm from Exp1. On the leaderboard the order is mean > Exp3 ≈ Exp1. The distance from fold 0 to the leaderboard is 0.005 for Exp1, 0.028 for Exp3 and 0.009 for the mean, so fold 0 does not track the leaderboard. Each pet has only 8–10 validation photos (one photo is about 0.012 macro F1) and both the head and the checkpoint of Exp3 were chosen on fold 0. We therefore compare models by the leaderboard here, and a five-fold evaluation would be the proper check.

**4. What we did not tune.** Equal weights, one threshold (0.25) for all labels, and no stacking. At 0.25 the mean made slightly more Cat and Dog false positives on fold 0 than either member, and its F1 at 0.5 was higher (0.99871), but a difference of one photo is not evidence. We cannot say whether other weights or per-label thresholds would add anything.